In [2]:
import pandas as pd
import numpy as np
from sklearn.cluster import MiniBatchKMeans

# 1. AYARLAR
# ---------------------------------------------------------
INPUT_FILE = "tum_hava_verileri_birlesik.csv"
OUTPUT_FILE = "clustered_weather_data.csv"

# Orijinal dosyadaki sütun sırası (Birebir aynı olması için)
ORIGINAL_COLUMNS = [
    'time', 'city_id', 'city_name', 'country', 
    'tavg', 'tmin', 'tmax', 'lat', 'lng', 'station_id'
]

# Hedeflenen nokta sayısı (Performans için)
TARGET_CLUSTERS = 2000  

# 2. VERİYİ YÜKLE
# ---------------------------------------------------------
print("Veri okunuyor...")
df = pd.read_csv(INPUT_FILE)
df['time'] = pd.to_datetime(df['time'])

# 3. KÜMELEME İŞLEMİ (CLUSTERING)
# ---------------------------------------------------------
unique_locations = df[['lat', 'lng']].drop_duplicates().reset_index(drop=True)
print(f"Toplam benzersiz lokasyon: {len(unique_locations)}")

if len(unique_locations) <= TARGET_CLUSTERS:
    print("Zaten az lokasyon var, olduğu gibi kaydediliyor.")
    df.to_csv(OUTPUT_FILE, index=False, float_format='%.4f')
else:
    print(f"Lokasyonlar {TARGET_CLUSTERS} kümeye indirgeniyor...")
    
    # K-Means ile grupla
    kmeans = MiniBatchKMeans(n_clusters=TARGET_CLUSTERS, random_state=42, batch_size=1024, n_init="auto")
    unique_locations['cluster_label'] = kmeans.fit_predict(unique_locations[['lat', 'lng']])
    
    # Merkez koordinatları hesapla
    cluster_centers = unique_locations.groupby('cluster_label')[['lat', 'lng']].mean().reset_index()
    cluster_centers.rename(columns={'lat': 'new_lat', 'lng': 'new_lng'}, inplace=True)

    # 4. VERİ BİRLEŞTİRME VE ÖZETLEME
    # ---------------------------------------------------------
    print("Veriler özetleniyor...")
    
    # Orijinal veriye cluster etiketini ekle
    df_merged = df.merge(unique_locations, on=['lat', 'lng'], how='left')
    
    # Aggregation kuralları (Orijinal yapıya sadık kalmak için)
    agg_funcs = {
        'tavg': 'mean',
        'tmin': 'mean',
        'tmax': 'mean',
        'city_name': 'first',    # Kümedeki ilk şehir ismini al
        'country': 'first',
        'city_id': 'first',      # ID yapısını koru
        'station_id': 'first'    # Station ID'yi de koru (varsa)
    }
    
    # 'station_id' dosyada yoksa hata vermesin diye kontrol
    if 'station_id' not in df.columns:
        # Eğer orijinalde yoksa listeden çıkar
        if 'station_id' in agg_funcs: del agg_funcs['station_id']
        if 'station_id' in ORIGINAL_COLUMNS: ORIGINAL_COLUMNS.remove('station_id')

    # Gruplama
    df_clustered = df_merged.groupby(['cluster_label', 'time']).agg(agg_funcs).reset_index()
    
    # Yeni koordinatları ekle
    df_clustered = df_clustered.merge(cluster_centers, on='cluster_label', how='left')
    
    # 5. FORMAT DÜZENLEME (EN ÖNEMLİ KISIM)
    # ---------------------------------------------------------
    
    # Koordinatları güncelle
    df_clustered['lat'] = df_clustered['new_lat']
    df_clustered['lng'] = df_clustered['new_lng']
    
    # Değerleri yuvarla (Orijinal CSV formatına benzesin)
    df_clustered['lat'] = df_clustered['lat'].round(4) # Koordinat hassasiyeti
    df_clustered['lng'] = df_clustered['lng'].round(4)
    df_clustered['tavg'] = df_clustered['tavg'].round(1)
    df_clustered['tmin'] = df_clustered['tmin'].round(1)
    df_clustered['tmax'] = df_clustered['tmax'].round(1)

    # Tarih formatını 'YYYY-MM-DD' yap
    df_clustered['time'] = df_clustered['time'].dt.strftime('%Y-%m-%d')

    # Sütunları orijinal sıraya diz
    df_final = df_clustered[ORIGINAL_COLUMNS]
    
    # 6. KAYDET
    # ---------------------------------------------------------
    print(f"Dosya kaydediliyor: {OUTPUT_FILE}")
    print(f"Satır sayısı: {len(df_final)}")
    
    df_final.to_csv(OUTPUT_FILE, index=False)
    print("✅ İşlem tamam. Dosya yapısı orijinaliyle birebir aynı.")

Veri okunuyor...


/var/folders/2h/58h_g_1n0lnfsq7v28ybqtr40000gn/T/ipykernel_32042/3807434719.py:22: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(INPUT_FILE)


Toplam benzersiz lokasyon: 47021
Lokasyonlar 2000 kümeye indirgeniyor...


/Users/emrecanulu/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/emrecanulu/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/emrecanulu/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/emrecanulu/Library/Python/3.9/lib/python/site-packages/sklearn/cluster/_kmeans.py:237: RuntimeWarning: divide by zero encountered in matmul
  current_pot = closest_dist_sq @ sample_weight
/Users/emrecanulu/Library/Python/3.9/lib/python/site-packages/sklearn/cluster/_kmeans.py:237: RuntimeWarning: overflow encountered in matmul
  current_pot = closest_dist_sq @ sample_weight
/Users/emrecanulu/Library/Python/3.9/lib/python/site-packages/sklearn/cluster/_kmeans.py:237: RuntimeWarning: invalid value encountered in matmul


Veriler özetleniyor...
Dosya kaydediliyor: clustered_weather_data.csv
Satır sayısı: 305822
✅ İşlem tamam. Dosya yapısı orijinaliyle birebir aynı.
